# 03 · Modeling — RF vs LR, interval, depreciation

Builds the pipeline, runs `GroupKFold(groups=model)`, and produces the Gate-2 evidence. Logic lives in `ml.train` / `ml.predict`.

In [ ]:
import sys; sys.path.insert(0, '..' if __import__('pathlib').Path('..').joinpath('app').exists() else '../..')
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from ml import ingest, train
from ml.predict import Predictor
sns.set_theme()
df = ingest.build_engineered_csv(); df.shape

In [ ]:
print('FEATURES:', train.FEATURES)
print('CATEGORICAL:', train.CATEGORICAL, ' NUMERIC:', train.NUMERIC)
print('TARGET:', train.TARGET, ' GROUP:', train.GROUP, ' RANDOM_STATE:', train.RANDOM_STATE)
print('interval band: [%.1f, %.1f] percentile' % (train.INTERVAL_LOW_PCT, train.INTERVAL_HIGH_PCT))

In [ ]:
prep = train.build_preprocessor()
Xt = prep.fit_transform(df[train.FEATURES])
feature_names = prep.get_feature_names_out()
print('transformed shape:', Xt.shape, f'({len(feature_names)} columns after one-hot + scaling)')
list(feature_names)

In [ ]:
for i, (tr_idx, va_idx) in enumerate(train.iter_group_folds(df, n_splits=5)):
    tr_models = set(df.iloc[tr_idx]['model']); va_models = set(df.iloc[va_idx]['model'])
    print(f'fold {i}: train={len(tr_idx)} rows/{len(tr_models)} models  '
          f'val={len(va_idx)} rows/{len(va_models)} models  val models={sorted(va_models)}')

In [ ]:
meta = train.main(df=df)  # writes artifacts + returns the metrics dict
rf = meta['models']['random_forest']['aggregate']; lr = meta['models']['linear_regression']['aggregate']
pd.DataFrame({'RandomForest': {k: v['mean'] for k,v in rf.items()},
              'LinearRegression': {k: v['mean'] for k,v in lr.items()}})

In [ ]:
print('interval:', meta['interval'])

### Per-fold breakdown & metric sanity check
`train.main()` above only surfaces the aggregate (mean +/- std) table -- re-running
`train.evaluate()` directly exposes the per-fold numbers so you can see fold-to-fold variance,
and `ml.metrics` functions are sanity-checked standalone against hand-computed values.

In [ ]:
report = train.evaluate(df, n_splits=5, n_estimators=300)  # same call train.main() makes internally
folds_rf = pd.DataFrame(report['random_forest']['folds'])
folds_lr = pd.DataFrame(report['linear_regression']['folds'])
print('RF per-fold:'); display(folds_rf)
print('LR per-fold:'); display(folds_lr)
folds_rf.plot(marker='o', title='RF metric variance across folds'); plt.show()

In [ ]:
import numpy as np
from ml import metrics
y_true = np.array([10_000., 20_000., 30_000.])
y_pred = np.array([9_000., 21_000., 33_000.])
print('mae', metrics.mae(y_true, y_pred))
print('rmse', metrics.rmse(y_true, y_pred))
print('r2', metrics.r2(y_true, y_pred))
print(f'mape (floor={metrics.MAPE_FLOOR_RM:.0f})', metrics.mape(y_true, y_pred))
print('evaluate_metrics ->', metrics.evaluate_metrics(y_true, y_pred))

### Feature importances -- which signals is the RF actually using?
This is the key view for "does the model learn the right features": if `battery_soh` or
`trans_adapt_offset` rank near the bottom, the engineered signal isn't adding value and the
formulas in `ml/features.py` are worth revisiting.

In [ ]:
import joblib
rf_pipe = joblib.load(train.ARTIFACTS_DIR / 'model.joblib')  # the production pipeline train.main() just wrote
prep2, rf2 = rf_pipe.named_steps['prep'], rf_pipe.named_steps['model']
importances = pd.Series(rf2.feature_importances_, index=prep2.get_feature_names_out()).sort_values(ascending=False)
importances.head(20)

In [ ]:
importances.head(20).plot.barh(figsize=(6, 6)); plt.gca().invert_yaxis()
plt.title('RF feature importances (top 20)'); plt.xlabel('importance'); plt.tight_layout()

### Experiment: hyperparameter tuning harness (no edits to `ml/train.py` needed)
Change `n_estimators` / `max_depth` / `min_samples_leaf` / `max_features` below and re-run
to compare against the production RF aggregate table above.

In [ ]:
# Quick experiment: try new hyperparameters WITHOUT editing ml/train.py.
# Swap the values below and re-run this cell to compare against the production
# aggregate table above. `train.make_pipeline`/`train.iter_group_folds` are the
# exact same building blocks train.evaluate() uses internally.
from sklearn.ensemble import RandomForestRegressor
experiment_rf = RandomForestRegressor(
    n_estimators=500, max_depth=15, min_samples_leaf=4,
    max_features='sqrt', random_state=train.RANDOM_STATE, n_jobs=-1,
)
experiment_pipe = train.make_pipeline(experiment_rf)
exp_fold_metrics = []
for tr_idx, va_idx in train.iter_group_folds(df, n_splits=5):
    experiment_pipe.fit(df.iloc[tr_idx][train.FEATURES], df.iloc[tr_idx][train.TARGET])
    preds = experiment_pipe.predict(df.iloc[va_idx][train.FEATURES])
    exp_fold_metrics.append(metrics.evaluate_metrics(df.iloc[va_idx][train.TARGET], preds))
print('experiment aggregate (mean over folds):')
pd.DataFrame(exp_fold_metrics).mean()

### Sanity predictions

In [ ]:
pred = Predictor()
profile = {'model':'C Class','year':2018,'age':8,'mileage':60000,'transmission':'Automatic',
           'fuel_type':'Petrol','engine_size':2.0,'mpg':45.0,'tax':150.0}
print(pred.predict(profile))

In [ ]:
# Visualize the per-tree spread behind predict()'s interval (Predictor._per_tree)
trees = pred._per_tree(pred._row(profile))
plt.hist(trees, bins=30)
for pct, color in [(train.INTERVAL_LOW_PCT, 'red'), (50, 'black'), (train.INTERVAL_HIGH_PCT, 'red')]:
    plt.axvline(np.percentile(trees, pct), color=color, ls='--')
plt.title(f'{len(trees)} tree predictions for this profile'); plt.xlabel('price_rm'); plt.show()

### Depreciation curve

In [ ]:
pts = pred.depreciation(profile, years=6)
curve = pd.DataFrame(pts)
sns.lineplot(data=curve, x='year', y='retained_pct', marker='o'); plt.title('retained value'); curve

In [ ]:
# module-level predict()/depreciation() -- these load the default artifacts/model.joblib on disk
from ml import predict as predict_module
print(predict_module.predict(profile))
print(predict_module.depreciation(profile, years=3))

### Experiment loop
1. Tweak `ml/features.py` formulas -> re-run `02_cleaning.ipynb` (regenerates the engineered CSV).
2. Tweak `ml/train.py` hyperparameters (or use the tuning harness above) -> re-run this notebook.
3. Check the feature-importances plot and the RF-vs-LR table above -- did the change help?
4. Once satisfied, update the real defaults in `ml/train.py` (`build_rf`, `main`) and re-run
   `python -m ml.train` from `backend/` to refresh the production `model.joblib` + `metrics.json`.

**Gate 2:** review the RF-vs-LR table, the empirical interval coverage, and these sanity predictions before Phase 03 wires the artifact into the dashboard.